<a href="https://colab.research.google.com/github/oyatillonewuu/dblabs/blob/main/lab12/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!sudo apt update
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
#Check this site for the latest download link https://www.apache.org/dyn/closer.lua/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!wget -q https://dlcdn.apache.org/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!tar xf spark-3.2.1-bin-hadoop3.2.tgz
!pip install -q findspark
!pip install pyspark
!pip install py4j

import os
import sys
# os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
# os.environ["SPARK_HOME"] = "/content/spark-3.2.1-bin-hadoop3.2"


import findspark
findspark.init()
findspark.find()

import pyspark

from pyspark.sql import DataFrame, SparkSession
from typing import List
import pyspark.sql.types as T
import pyspark.sql.functions as F

spark= SparkSession \
       .builder \
       .appName("Our First Spark Example") \
       .getOrCreate()

spark

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,968 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.0 MB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,943 kB]
Get:14

In [10]:
text_rdd = spark.sparkContext.parallelize([
    "do one",
    "do two",
    "do three",
    "map all",
    "reduce all"
])
words_rdd = text_rdd.flatMap(
    lambda line: line.split(" ")
).map(lambda word: (word, 1))
print(words_rdd.collect())

word_counts = words_rdd.reduceByKey(lambda a, b: a + b)
print(word_counts.collect())

[('do', 1), ('one', 1), ('do', 1), ('two', 1), ('do', 1), ('three', 1), ('map', 1), ('all', 1), ('reduce', 1), ('all', 1)]
[('three', 1), ('map', 1), ('do', 3), ('one', 1), ('two', 1), ('all', 2), ('reduce', 1)]


In [19]:
import json

results = word_counts.collect()
result_dict = {word: count for word, count in results}

with open("/content/drive/MyDrive/Colab Notebooks/files/lab12/wordcount.json", "w") as f:
  json.dump(result_dict, f)

In [18]:
# helper functions

def showNumPartitions(data):
  print(f"Number of partitions: ", data.getNumPartitions())

def show_partition(p_id, p_data):
  return [(p_id, list(p_data)[:5])]

data = spark.sparkContext.parallelize(range(100), numSlices=4)
part_info= data.mapPartitionsWithIndex(show_partition).collect()
print(part_info)

[(0, [0, 1, 2, 3, 4]), (1, [25, 26, 27, 28, 29]), (2, [50, 51, 52, 53, 54]), (3, [75, 76, 77, 78, 79])]
